# Berlin building classification -- raw-polygon Transformer encoder (Path E)

This is the literal architecture described on the resume: classifies building type
**directly from raw polygon geometry** -- no engineered features, no spatial graph. Each
building's boundary is a variable-length sequence of (x, y) points; a standard
`nn.TransformerEncoder` processes that sequence with a padding-aware attention mask, and a
pooled representation feeds a classification head.

Distinct from `berlin_graph_transformer.ipynb` (Path D, the Graph Transformer over 86
engineered tabular features with a k-NN spatial graph) -- that one stays as its own result.
This is a different, independent model, trained and evaluated the same way for comparability.

**Why this doesn't have the graph model's OOM problem**: there's no city-wide graph and no
O(n^2)/O(edges) structure here -- each building is an independent short sequence (padded to
32 points, picked from the real p95 of vertex counts), processed in ordinary mini-batches.
Memory scales with batch size, not dataset size, so this trains on the full 331k-building
labeled set directly, no subsampling needed.

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path("..").resolve()))

import numpy as np
import geopandas as gpd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cpu


In [2]:
buildings = gpd.read_parquet("../data/processed/berlin_buildings_features.parquet", columns=["geometry", "alkis_label"])
labeled = buildings.dropna(subset=["alkis_label"]).reset_index(drop=True)
print(len(labeled), "labeled buildings")
print(labeled.geom_type.value_counts().to_dict())

331168 labeled buildings
{'Polygon': 331145, 'MultiPolygon': 23}


In [3]:
# Geometry extraction. MAX_LEN=32 covers ~p95-p97 of observed exterior-ring vertex counts
# (median 7, p90 17, p95 23, p99 41, max 217 on a 20k sample) -- longer rings get truncated,
# a small, explicit information loss on a handful of unusually complex footprints, in exchange
# for a fixed, cheap sequence length. The 23 MultiPolygon buildings (out of 331,168) take their
# largest sub-polygon by area.
MAX_LEN = 32

def exterior_coords(geom):
    if geom.geom_type == "MultiPolygon":
        geom = max(geom.geoms, key=lambda g: g.area)
    pts = np.asarray(geom.exterior.coords, dtype=np.float32)
    return pts[:-1]  # drop the repeated closing point (first == last in a shapely ring)

n = len(labeled)
coords_padded = np.zeros((n, MAX_LEN, 2), dtype=np.float32)
mask = np.zeros((n, MAX_LEN), dtype=bool)  # True = real point, False = padding

for i, geom in enumerate(labeled.geometry.values):
    pts = exterior_coords(geom)
    L = min(len(pts), MAX_LEN)
    coords_padded[i, :L] = pts[:L]
    mask[i, :L] = True

y_raw = (labeled["alkis_label"] == "non_residential").to_numpy(dtype=np.int64)
print("coords_padded:", coords_padded.shape, "mask:", mask.shape)
print("mean real points per building:", mask.sum(axis=1).mean())
print("class balance:", np.bincount(y_raw))

coords_padded: (331168, 32, 2) mask: (331168, 32)
mean real points per building: 8.45295741134409
class balance: [266121  65047]


In [4]:
# Coordinate normalization: center each polygon on its OWN centroid (translation invariance --
# a building's absolute position in Berlin shouldn't matter, only its shape and size), then
# divide by a single GLOBAL constant rather than a per-building scale. A per-building unit-scale
# normalization (common in point-cloud/shape learning) would throw away absolute size -- and
# footprint area is one of the single strongest predictors in every other model in this project
# (top-2 feature importance in clf2_berlin). Global scaling keeps relative size differences
# between buildings intact while still putting coordinates in a numerically stable range.
SCALE_M = 50.0  # meters; most Berlin building footprints span well under this per side

sum_xy = (coords_padded * mask[..., None]).sum(axis=1)
counts = mask.sum(axis=1, keepdims=True).clip(min=1)
centroids = sum_xy / counts  # (n, 2), computed over real points only

coords_norm = coords_padded - centroids[:, None, :]
coords_norm[~mask] = 0.0  # re-zero padding positions (subtracting the centroid moved them off zero)
coords_norm = (coords_norm / SCALE_M).astype(np.float32)

print("normalized coord range:", coords_norm[mask].min(), "to", coords_norm[mask].max())

normalized coord range: -5.7725 to 4.858125


In [5]:
# Stratified 70/10/20 train/val/test split, same pattern as the graph transformer notebook.
idx = np.arange(n)
train_idx, test_idx = train_test_split(idx, test_size=0.2, random_state=42, stratify=y_raw)
train_idx, val_idx = train_test_split(train_idx, test_size=0.1, random_state=42, stratify=y_raw[train_idx])

coords_t = torch.from_numpy(coords_norm)
mask_t = torch.from_numpy(mask)
y_t = torch.from_numpy(y_raw)

def make_loader(indices, batch_size, shuffle):
    ds = TensorDataset(coords_t[indices], mask_t[indices], y_t[indices])
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

BATCH_SIZE = 512
train_loader = make_loader(train_idx, BATCH_SIZE, shuffle=True)
val_loader = make_loader(val_idx, BATCH_SIZE, shuffle=False)
test_loader = make_loader(test_idx, BATCH_SIZE, shuffle=False)
print("train/val/test:", len(train_idx), len(val_idx), len(test_idx))

train/val/test: 238440 26494 66234


In [6]:
class PolygonTransformerEncoder(nn.Module):
    """Raw-polygon Transformer encoder: (x, y) boundary points -> learned embedding + learned
    positional embedding -> nn.TransformerEncoder with a padding mask (attention never attends
    to padded positions) -> mean-pool over real positions -> classify.

    Positional embedding caveat worth knowing: a polygon ring is a *cyclic* sequence with an
    arbitrary starting vertex (shapely's ring order is a digitization artifact, not a
    meaningful "first point"), so a standard sequence positional embedding is a slight mismatch
    for this data -- it's included because it's standard Transformer practice and directly
    matches what "a Transformer encoder" implies, not because ring order is semantically
    meaningful here the way word order is in a sentence.
    """
    def __init__(self, d_model=32, nhead=4, num_layers=2, max_len=MAX_LEN, num_classes=2, dropout=0.1):
        super().__init__()
        self.input_proj = nn.Linear(2, d_model)
        self.pos_embed = nn.Parameter(torch.zeros(1, max_len, d_model))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model * 4,
            dropout=dropout, batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.classifier = nn.Linear(d_model, num_classes)

    def forward(self, coords, real_mask):
        # coords: (B, L, 2) float, real_mask: (B, L) bool, True = real point
        x = self.input_proj(coords) + self.pos_embed
        pad_mask = ~real_mask  # nn.TransformerEncoder wants True = IGNORE this position
        h = self.encoder(x, src_key_padding_mask=pad_mask)
        mask_f = real_mask.unsqueeze(-1).float()
        pooled = (h * mask_f).sum(dim=1) / mask_f.sum(dim=1).clamp(min=1)
        return self.classifier(pooled)


model = PolygonTransformerEncoder().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

# Same 2:1 non_residential:residential class weighting used throughout this project.
class_weights = torch.tensor([1.0, 2.0], dtype=torch.float, device=device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
print(model)
print(sum(p.numel() for p in model.parameters()), "parameters")

PolygonTransformerEncoder(
  (input_proj): Linear(in_features=2, out_features=32, bias=True)
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=32, out_features=32, bias=True)
        )
        (linear1): Linear(in_features=32, out_features=128, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=128, out_features=32, bias=True)
        (norm1): LayerNorm((32,), eps=1e-05, elementwise_affine=True, bias=True)
        (norm2): LayerNorm((32,), eps=1e-05, elementwise_affine=True, bias=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (classifier): Linear(in_features=32, out_features=2, bias=True)
)
26594 parameters


In [7]:
def run_epoch(loader, train):
    model.train(mode=train)
    total_loss, all_pred, all_true = 0.0, [], []
    for coords, m, labels in loader:
        coords, m, labels = coords.to(device), m.to(device), labels.to(device)
        with torch.set_grad_enabled(train):
            logits = model(coords, m)
            loss = criterion(logits, labels)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
        total_loss += loss.item() * len(labels)
        all_pred.append(logits.argmax(dim=1).detach().cpu())
        all_true.append(labels.cpu())
    all_pred = torch.cat(all_pred).numpy()
    all_true = torch.cat(all_true).numpy()
    acc = accuracy_score(all_true, all_pred)
    f1_macro = f1_score(all_true, all_pred, average="macro")
    return total_loss / len(all_true), acc, f1_macro


EPOCHS = 15
for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc, train_f1 = run_epoch(train_loader, train=True)
    val_loss, val_acc, val_f1 = run_epoch(val_loader, train=False)
    print(f"epoch {epoch:2d} | train loss {train_loss:.4f} acc {train_acc:.4f} | "
          f"val loss {val_loss:.4f} acc {val_acc:.4f} macro_f1 {val_f1:.4f}")

/home/labdoo/anaconda3/envs/berlin-buildings/lib/python3.11/site-packages/torch/nn/modules/transformer.py:529: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/NestedTensorImpl.cpp:177.)
  output = torch._nested_tensor_from_mask(


epoch  1 | train loss 0.5479 acc 0.8181 | val loss 0.4836 acc 0.8397 macro_f1 0.7334


epoch  2 | train loss 0.4842 acc 0.8335 | val loss 0.4741 acc 0.8301 macro_f1 0.7379


epoch  3 | train loss 0.4688 acc 0.8377 | val loss 0.4565 acc 0.8468 macro_f1 0.7513


epoch  4 | train loss 0.4615 acc 0.8413 | val loss 0.4608 acc 0.8542 macro_f1 0.7513


epoch  5 | train loss 0.4581 acc 0.8434 | val loss 0.4538 acc 0.8540 macro_f1 0.7549


... (6 outputs left out)

epoch 12 | train loss 0.4472 acc 0.8481 | val loss 0.4437 acc 0.8568 macro_f1 0.7628


epoch 13 | train loss 0.4466 acc 0.8487 | val loss 0.4453 acc 0.8490 macro_f1 0.7598


epoch 14 | train loss 0.4466 acc 0.8477 | val loss 0.4464 acc 0.8575 macro_f1 0.7617


epoch 15 | train loss 0.4451 acc 0.8489 | val loss 0.4420 acc 0.8511 macro_f1 0.7625


In [8]:
model.eval()
all_pred, all_true = [], []
with torch.no_grad():
    for coords, m, labels in test_loader:
        coords, m = coords.to(device), m.to(device)
        logits = model(coords, m)
        all_pred.append(logits.argmax(dim=1).cpu())
        all_true.append(labels)
all_pred = torch.cat(all_pred).numpy()
all_true = torch.cat(all_true).numpy()

print("Accuracy:", accuracy_score(all_true, all_pred))
print(classification_report(all_true, all_pred, target_names=["residential", "non_residential"]))
print(confusion_matrix(all_true, all_pred))

print("\nFor reference:")
print("clf2_berlin (DecisionTree, engineered features, 331k):  Accuracy 0.9189, non-res F1 0.79, macro F1 0.87")
print("Graph Transformer (TransformerConv, engineered features + k-NN graph, 60k subsample): Accuracy 0.9106, non-res F1 0.76, macro F1 0.85")

Accuracy: 0.8539420841259776
                 precision    recall  f1-score   support

    residential       0.91      0.91      0.91     53225
non_residential       0.63      0.62      0.62     13009

       accuracy                           0.85     66234
      macro avg       0.77      0.76      0.77     66234
   weighted avg       0.85      0.85      0.85     66234

[[48524  4701]
 [ 4973  8036]]

For reference:
clf2_berlin (DecisionTree, engineered features, 331k):  Accuracy 0.9189, non-res F1 0.79, macro F1 0.87
Graph Transformer (TransformerConv, engineered features + k-NN graph, 60k subsample): Accuracy 0.9106, non-res F1 0.76, macro F1 0.85
